# Update Author Full Names

Maintains the `full_name` column on the base identity table `openalex.authors.authors` (catalog
`openalex`, schema `authors`, table `authors` — not to be confused with the enriched `openalex_authors`).
full_name is the matching name behind `block_key` and the parsed name columns in `authors_for_matching`. Historically written once at author creation and never updated.
There is no direct full_name curation: candidates are attested raw names from the author's works
(previous run's `author_works_staging`) plus the curated display_name, all passing the same guards.

Runs in `jobs/authors.yaml` before `Create_Authors`, so changes land in the same day's rebuild. ES sync
is automatic (full_name is in the CreateAuthors content hash); no work re-sync needed.

Change reasons:
- **enrich** — attested variant that strictly refines the current full_name: parsed last + first-initial
  binary-equal (parser output is lowercased/diacritic-stripped, so block_key provably unchanged); first
  may go initial→word, middle may appear or initial→word, suffix may appear, nothing degrades. Needs ≥2
  works and no conflicting middle/first evidence among supported variants.
- **enrich_curated** — the curated display_name as candidate, same refinement rules; human assertion
  bypasses the support gate and conflict vetoes, and zero-gain format fixes are allowed
  (`Valko, Michal` → `Michal Valko`). A richer attested variant outranks a format-only curation.
- **pollution_reset / fill_missing** — full_name's parsed last matches no attested raw (wrong block),
  or full_name is missing/unparseable: reset to the dominant attested raw (≥3 works, ≥50% share).
  Changes block_key by design.

Safety: every adopted name is attested in `author_names` (parse guaranteed); preflight cell aborts on any
block_key drift in enrich rows, malformed/duplicate rows, or per-reason volume caps. Idempotent — re-run
after apply yields zero changes. `author_full_name_pending_changes` persists until the next run for QA.
Validated on a 0.1% sample 2026-06-10: ~90K enrich, ~190K pollution_reset (legacy PubMed `Lastname Xy`
forms in wrong blocks), ~2K fill_missing expected on first run.

## Candidate evidence

One row per (author, candidate name): attested raw names with work counts, plus curated display_names
(`is_curated`, `n` NULL unless also attested). Inner join to `author_names` guarantees a parse with a
non-empty last name. Materialized because the change computation reads it four times.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_full_name_evidence
CLUSTER BY (author_id) AS
WITH attested AS (
  SELECT
    CAST(REPLACE(s.author_id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
    TRIM(s.raw_author_name) AS raw,
    COUNT(*) AS n,
    FALSE AS is_curated
  FROM openalex.authors.author_works_staging s
  WHERE s.raw_author_name IS NOT NULL
    AND TRIM(s.raw_author_name) <> ''
  GROUP BY 1, 2
),
curated AS (
  SELECT
    c.author_id,
    TRIM(c.curated_display_name) AS raw,
    CAST(NULL AS BIGINT) AS n,
    TRUE AS is_curated
  FROM openalex.authors.author_names_curations c
  WHERE c.curated_display_name IS NOT NULL
    AND TRIM(c.curated_display_name) <> ''
),
unioned AS (
  SELECT author_id, raw, SUM(n) AS n, BOOL_OR(is_curated) AS is_curated
  FROM (SELECT * FROM attested UNION ALL SELECT * FROM curated)
  GROUP BY author_id, raw
)
SELECT
  u.author_id,
  u.raw,
  u.n,
  u.is_curated,
  p.parsed_name.first AS p_first,
  COALESCE(p.parsed_name.middle, '') AS p_middle,
  p.parsed_name.last AS p_last,
  COALESCE(p.parsed_name.suffix, '') AS p_suffix,
  (u.raw NOT RLIKE '[0-9,;:()\\[\\]{}<>/@&#%*_=+|"!?]'
   AND LENGTH(u.raw) BETWEEN 5 AND 80
   AND u.raw LIKE '% %'
   AND COALESCE(p.parsed_name.first, '') <> ''
  ) AS is_clean
FROM unioned u
JOIN openalex.authors.author_names p ON u.raw = p.raw_author_name
WHERE p.parsed_name.last IS NOT NULL
  AND p.parsed_name.last <> ''

## Stage pending changes

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_full_name_pending_changes AS
WITH current AS (
  SELECT
    a.id AS author_id,
    TRIM(a.full_name) AS full_name,
    p.parsed_name.first AS c_first,
    COALESCE(p.parsed_name.middle, '') AS c_middle,
    p.parsed_name.last AS c_last,
    COALESCE(p.parsed_name.suffix, '') AS c_suffix
  FROM openalex.authors.authors a
  LEFT JOIN openalex.authors.author_names p
    ON TRIM(a.full_name) = p.raw_author_name
),

-- per-author rollups over all attested evidence (for the reset paths)
evidence_stats AS (
  SELECT
    author_id,
    SUM(n) AS total_n,
    MAX_BY(STRUCT(raw, n), STRUCT(n, LENGTH(raw), raw)) FILTER (WHERE is_clean) AS top_clean
  FROM openalex.authors.author_full_name_evidence
  WHERE n IS NOT NULL
  GROUP BY author_id
),
last_match AS (
  SELECT e.author_id,
         MAX(CASE WHEN e.p_last = c.c_last THEN 1 ELSE 0 END) AS any_last_match
  FROM openalex.authors.author_full_name_evidence e
  JOIN current c ON e.author_id = c.author_id
  WHERE e.n IS NOT NULL
  GROUP BY e.author_id
),

-- enrich path: variants sharing the current full_name's block-key components
compat AS (
  SELECT
    e.author_id, e.raw, e.n, e.is_clean, e.is_curated,
    e.p_suffix, c.full_name, c.c_suffix,
    REGEXP_REPLACE(c.c_first, '[^\\p{L}]', '')  AS c_first_letters,
    REGEXP_REPLACE(e.p_first, '[^\\p{L}]', '')  AS e_first_letters,
    REGEXP_REPLACE(c.c_middle, '[^\\p{L}]', '') AS c_mid_letters,
    REGEXP_REPLACE(e.p_middle, '[^\\p{L}]', '') AS e_mid_letters
  FROM openalex.authors.author_full_name_evidence e
  JOIN current c ON e.author_id = c.author_id
  WHERE c.c_last IS NOT NULL AND c.c_last <> ''
    AND e.p_last = c.c_last
    AND e.p_first IS NOT NULL AND e.p_first <> ''
    AND c.c_first IS NOT NULL AND c.c_first <> ''
    AND SUBSTRING(e.p_first, 1, 1) = SUBSTRING(c.c_first, 1, 1)
),
flagged AS (
  SELECT *,
    -- per-component gain: 0 = equal, 1 = strict refinement, NULL = degradation/conflict (disqualifies)
    CASE WHEN e_first_letters = c_first_letters THEN 0
         WHEN LENGTH(c_first_letters) <= 1 AND LENGTH(e_first_letters) >= 3 THEN 1
         ELSE NULL END AS first_gain,
    CASE WHEN e_mid_letters = c_mid_letters THEN 0
         WHEN c_mid_letters = '' AND e_mid_letters <> '' THEN 1
         WHEN LENGTH(c_mid_letters) = 1 AND LENGTH(e_mid_letters) >= 3
              AND SUBSTRING(e_mid_letters, 1, 1) = SUBSTRING(c_mid_letters, 1, 1) THEN 1
         ELSE NULL END AS middle_gain,
    CASE WHEN e_suffix_eq THEN 0
         WHEN c_suffix = '' AND p_suffix <> '' THEN 1
         ELSE NULL END AS suffix_gain
  FROM (SELECT *, COALESCE(p_suffix, '') = COALESCE(c_suffix, '') AS e_suffix_eq FROM compat)
),
conflicts AS (
  SELECT author_id,
    COUNT(DISTINCT CASE WHEN n >= 2 AND e_mid_letters <> ''
                        THEN SUBSTRING(e_mid_letters, 1, 1) END) AS distinct_mid_initials,
    COUNT(DISTINCT CASE WHEN n >= 2 AND LENGTH(e_first_letters) >= 3
                        THEN e_first_letters END) AS distinct_full_firsts
  FROM flagged
  GROUP BY author_id
),
enrich AS (
  SELECT author_id, old_full_name, new_full_name, reason, support_n
  FROM (
    SELECT
      f.author_id,
      f.full_name AS old_full_name,
      f.raw AS new_full_name,
      CASE WHEN f.is_curated THEN 'enrich_curated' ELSE 'enrich' END AS reason,
      f.n AS support_n,
      ROW_NUMBER() OVER (
        PARTITION BY f.author_id
        ORDER BY (f.first_gain + f.middle_gain + f.suffix_gain) DESC,
                 f.is_curated DESC,
                 f.n DESC, LENGTH(f.raw) DESC, f.raw ASC
      ) AS rk
    FROM flagged f
    JOIN conflicts cf ON f.author_id = cf.author_id
    WHERE f.is_clean
      AND (f.n >= 2 OR f.is_curated)
      AND f.first_gain IS NOT NULL AND f.middle_gain IS NOT NULL AND f.suffix_gain IS NOT NULL
      AND (f.first_gain + f.middle_gain + f.suffix_gain) >= (CASE WHEN f.is_curated THEN 0 ELSE 1 END)
      AND (f.first_gain = 0 OR f.is_curated OR cf.distinct_full_firsts <= 1)
      AND (f.middle_gain = 0 OR f.is_curated OR cf.distinct_mid_initials <= 1)
      AND f.raw <> f.full_name
  )
  WHERE rk = 1
),

-- reset paths: dominant attested raw replaces a missing or polluted full_name
resets AS (
  SELECT
    c.author_id,
    c.full_name AS old_full_name,
    es.top_clean.raw AS new_full_name,
    CASE WHEN c.full_name IS NULL OR c.full_name = '' OR c.c_last IS NULL OR c.c_last = ''
         THEN 'fill_missing' ELSE 'pollution_reset' END AS reason,
    es.top_clean.n AS support_n
  FROM current c
  JOIN evidence_stats es ON c.author_id = es.author_id
  LEFT JOIN last_match lm ON c.author_id = lm.author_id
  WHERE es.top_clean IS NOT NULL
    AND es.top_clean.n >= 3
    AND es.top_clean.n / es.total_n >= 0.5
    AND (
      c.full_name IS NULL OR c.full_name = ''
      OR c.c_last IS NULL OR c.c_last = ''
      OR COALESCE(lm.any_last_match, 0) = 0
    )
    AND es.top_clean.raw IS DISTINCT FROM c.full_name
)

SELECT * FROM enrich
UNION ALL
SELECT * FROM resets

## Preflight: abort before applying anything suspect

In [ ]:
%sql
WITH counts AS (
  SELECT
    COUNT_IF(reason LIKE 'enrich%') AS enrich_n,
    COUNT_IF(reason = 'pollution_reset') AS pollution_n,
    COUNT_IF(reason = 'fill_missing') AS missing_n,
    COUNT_IF(new_full_name IS NULL OR TRIM(new_full_name) = ''
             OR new_full_name IS NOT DISTINCT FROM old_full_name) AS bad_rows,
    COUNT(*) - COUNT(DISTINCT author_id) AS dup_authors
  FROM openalex.authors.author_full_name_pending_changes
),
block_key_drift AS (
  SELECT COUNT(*) AS n
  FROM openalex.authors.author_full_name_pending_changes pc
  JOIN openalex.authors.author_names po ON TRIM(pc.old_full_name) = po.raw_author_name
  JOIN openalex.authors.author_names pn ON TRIM(pc.new_full_name) = pn.raw_author_name
  WHERE pc.reason LIKE 'enrich%'
    AND (
      CASE WHEN po.parsed_name.last IS NULL THEN NULL
           WHEN po.parsed_name.first IS NULL OR po.parsed_name.first = '' THEN po.parsed_name.last
           ELSE CONCAT(SUBSTRING(po.parsed_name.first, 1, 1), ' ', po.parsed_name.last) END
      IS DISTINCT FROM
      CASE WHEN pn.parsed_name.last IS NULL THEN NULL
           WHEN pn.parsed_name.first IS NULL OR pn.parsed_name.first = '' THEN pn.parsed_name.last
           ELSE CONCAT(SUBSTRING(pn.parsed_name.first, 1, 1), ' ', pn.parsed_name.last) END
    )
)
SELECT CASE
  WHEN b.n > 0 THEN
    RAISE_ERROR(CONCAT('ABORT: ', CAST(b.n AS STRING), ' enrich changes would alter block_key'))
  WHEN c.bad_rows > 0 THEN
    RAISE_ERROR(CONCAT('ABORT: ', CAST(c.bad_rows AS STRING), ' pending rows are malformed'))
  WHEN c.dup_authors > 0 THEN
    RAISE_ERROR(CONCAT('ABORT: ', CAST(c.dup_authors AS STRING), ' authors have multiple pending changes'))
  WHEN c.enrich_n > 5000000 THEN
    RAISE_ERROR(CONCAT('ABORT: enrich count ', CAST(c.enrich_n AS STRING), ' exceeds cap 5,000,000'))
  WHEN c.pollution_n > 500000 THEN
    RAISE_ERROR(CONCAT('ABORT: pollution_reset count ', CAST(c.pollution_n AS STRING), ' exceeds cap 500,000'))
  WHEN c.missing_n > 100000 THEN
    RAISE_ERROR(CONCAT('ABORT: fill_missing count ', CAST(c.missing_n AS STRING), ' exceeds cap 100,000'))
  ELSE CONCAT('OK: enrich=', CAST(c.enrich_n AS STRING),
              ' pollution_reset=', CAST(c.pollution_n AS STRING),
              ' fill_missing=', CAST(c.missing_n AS STRING))
END AS preflight
FROM counts c CROSS JOIN block_key_drift b

## Apply

In [ ]:
%sql
MERGE INTO openalex.authors.authors AS target
USING (
  SELECT author_id, new_full_name
  FROM openalex.authors.author_full_name_pending_changes
  QUALIFY ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY support_n DESC, reason) = 1
) AS source
ON target.id = source.author_id
WHEN MATCHED THEN UPDATE SET target.full_name = source.new_full_name

## Spot-check

In [ ]:
%sql
SELECT *
FROM openalex.authors.author_full_name_pending_changes
QUALIFY ROW_NUMBER() OVER (PARTITION BY reason ORDER BY support_n DESC NULLS LAST) <= 25
ORDER BY reason, support_n DESC NULLS LAST
LIMIT 100